In [1]:
from torch.overrides import has_torch_function_unary, handle_torch_function
import torch
from torch import nn
class MyTensor(torch.Tensor):
    def __new__(cls,data: torch.Tensor, *args, **kwargs):
        return torch.Tensor._make_subclass(cls,data, *args, *kwargs)

    def __torch_function__(cls, func, types, args=(), kwargs=None):
        print(f"function: {getattr(func,"__self__",None) or func}")
                # with _C.DisableTorchFunctionSubclass():
            # ret = func(*args, **kwargs)
        return super().__torch_function__(func, types, args, kwargs)
    # __torch_function__ = torch._C._disabled_torch_function_impl

    def __torch_dispatch__(self,func, types, args=(), kwargs=None ):
        print(f"dispatch: {getattr(func,"__self__",None) or func}")
        kwargs = kwargs or {}
        return func(*args, **kwargs)

    def __repr__(self, *, tensor_contents=None):
        return f"MyTensor contain {super().__repr__(tensor_contents=tensor_contents)}"

In [2]:
t = MyTensor(torch.randn([3,3]))

In [ ]:
s = torch.add(t,5)

In [93]:
s.device

device(type='cpu')

In [94]:
lm = nn.Linear(64,32)
lm.weight = nn.Parameter(MyTensor(lm.weight.data))

In [95]:
input = torch.randn([2,64])
output = lm(input)

In [96]:
output.shape

torch.Size([2, 32])

In [50]:
type(s)

__main__.MyTensor